In [21]:
import fitz
import pytesseract
from PIL import Image


In [11]:
filepath = "PDFs\PV AG 2013 10 21 Démission du president et Nomination PIC.pdf"

In [22]:
doc = fitz.open(filepath)
tesseract_path = r"assets\tesseract\tesseract.exe"
pytesseract.pytesseract.tesseract_cmd = tesseract_path

text_parts = []

for page_num in range(len(doc)):
    pix = doc.load_page(page_num).get_pixmap()
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    text = pytesseract.image_to_string(img, lang="fra")
    text_parts.append(text)

text = "\n".join(text_parts)
print(text)

Brôûleur AEM
Société par actions simplifiée au capital de 250.000 €
Siège social : 6 rue des Nonettes 77500 CHELLES
RCS MEAUX 582 017 018

PROCES-VERBAL DES DECISIONS DE L'ASSOCIEE UNIQUE
EN DATE DU 21 OCTOBRE 2013

L'AN DEUX MILLE TRELZE ET LE
21 octobre à 10 heures au siège social,

La société P.LC, SAS au capital de 1.505.102 euros, dont le siège social est 4 rue Kuss
750013 PARIS, immatriculée sous le numéro RCS PARIS 510 952 305,

Seule associée de la Société Brôleur AEM,
{Après avoir pris connaissance du rapport du Président non associé,
A délibéré sur les points suivants +

— Constatation de la démission du Président et nomination de La société PIC, elle-
même représentée par M. Jean-Louis MARULAZ, en qualité de Président ;

— Résiliation de la convention de prestations de services entre les sociétés PIC et
Brûleur AEM ;

— Modification de l'amicle 17 des statuts relatif aux modalités de fixation de la
rémunération du Président ;

— Conclusion d’une convention de direction avec 

In [2]:
import re
import csv
from transformers import pipeline, AutoTokenizer
import collections
import os # Import the os module for file path operations

# --- Installation Note ---
# If you encounter a ValueError related to SentencePiece or Tiktoken,
# ensure you have these libraries installed:
# pip install transformers sentencepiece tiktoken
# Remember to restart your Python environment/kernel after installation.
# -------------------------

# Global variable for the NER pipeline and tokenizer to avoid re-loading on every call
_ner_pipeline = None
_tokenizer = None

def _initialize_ner_pipeline():
    """Initializes the global NER pipeline and tokenizer."""
    global _ner_pipeline, _tokenizer
    if _ner_pipeline is None or _tokenizer is None:
        model_name = "Jean-Baptiste/camembert-ner"
        try:
            # Load the tokenizer first, explicitly
            _tokenizer = AutoTokenizer.from_pretrained(model_name)
            # Then load the pipeline, passing the loaded tokenizer
            _ner_pipeline = pipeline(
                "ner",
                model=model_name,
                tokenizer=_tokenizer,
                aggregation_strategy="simple" # Aggregates multi-word entities like "Jean-Luc" into one
            )
            print(f"Successfully loaded NER model: {model_name}")
        except Exception as e:
            print(f"Error loading NER model: {e}")
            print("Please ensure 'transformers', 'sentencepiece', and 'tiktoken' are installed.")
            print("You might need to restart your Python environment after installation.")
            _ner_pipeline = None # Reset to None if loading fails
            _tokenizer = None
            raise # Re-raise the exception to inform the caller

def anonymize_text(text: str) -> tuple[str, dict]:
    """
    Detects person names in a given text and replaces them with anonymized tags (e.g., PERSON_1).

    Args:
        text (str): The input text in French.

    Returns:
        tuple[str, dict]: A tuple containing:
            - anonymized_text (str): The text with names replaced by anonymized tags.
            - anonymization_mapping (dict): A dictionary mapping anonymized tags to original names.
                                            (e.g., {"PERSON_1": "Emmanuel Macron"})
    """
    _initialize_ner_pipeline()
    if _ner_pipeline is None:
        raise RuntimeError("NER pipeline could not be initialized. Check previous error messages.")

    anonymization_mapping = {}
    person_counter = 0
    anonymized_text = list(text) # Convert to list for mutable character replacement

    # Get NER results
    ner_results = _ner_pipeline(text)

    # Collect entities and their replacement data
    # Iterate in reverse order of start index to avoid issues with index shifting
    # when replacing substrings in the `anonymized_text` list.
    replacements = []
    for entity in sorted(ner_results, key=lambda x: x['start'], reverse=True):
        if entity['entity_group'] == 'PER': # Check if it's a Person entity
            original_name = entity['word']
            start_index = entity['start']
            end_index = entity['end']

            # Check if this exact name has already been assigned an anonymized tag
            # We iterate through the existing mapping values to find the original name.
            # This handles cases where the same name appears multiple times in the text.
            assigned_tag = None
            for tag, name in anonymization_mapping.items():
                if name == original_name:
                    assigned_tag = tag
                    break

            if assigned_tag is None:
                # Assign a new anonymized tag if the name is encountered for the first time
                person_counter += 1
                anonymized_tag = f"PERSON_{person_counter}"
                anonymization_mapping[anonymized_tag] = original_name
            else:
                anonymized_tag = assigned_tag

            # Add to replacements list
            replacements.append((start_index, end_index, original_name, anonymized_tag))

    # Apply replacements from end to beginning
    current_text_list = list(text)
    for start, end, original_name, anon_tag in sorted(replacements, key=lambda x: x[0], reverse=True):
        current_text_list[start:end] = list(anon_tag)

    return "".join(current_text_list), anonymization_mapping


def deanonymize_text(anonymized_text: str, anonymization_mapping: dict) -> str:
    """
    Reverses the anonymization of names in a text using a provided mapping.

    Args:
        anonymized_text (str): The text with anonymized tags (e.g., "PERSON_1").
        anonymization_mapping (dict): The dictionary mapping anonymized tags to original names.
                                      (e.g., {"PERSON_1": "Emmanuel Macron"})

    Returns:
        str: The text with anonymized tags replaced by their original names.
    """
    deanonymized_text = anonymized_text

    # Sort the mapping keys (anonymized tags) by length in descending order
    # to prevent issues where a shorter tag (e.g., PERSON_1) might be a substring
    # of a longer tag (e.g., PERSON_10) and get replaced prematurely.
    # In this specific PERSON_X format, it's not strictly necessary as PERSON_1
    # is not a substring of PERSON_10 in a problematic way for simple replace,
    # but it's a good general practice for robust replacement.
    sorted_tags = sorted(anonymization_mapping.keys(), key=len, reverse=True)

    for tag in sorted_tags:
        original_name = anonymization_mapping[tag]
        # Use re.sub to replace all occurrences of the tag with the original name
        # We escape the tag in case it contains special regex characters, though PERSON_X won't.
        deanonymized_text = re.sub(re.escape(tag), original_name, deanonymized_text)

    return deanonymized_text

def save_mapping_to_csv(mapping: dict, filename: str):
    """
    Saves the anonymization mapping to a CSV file.

    Args:
        mapping (dict): The dictionary mapping anonymized tags to original names.
        filename (str): The path to the CSV file to save the mapping.
    """
    try:
        with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
            fieldnames = ['anonymized_tag', 'original_name']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

            writer.writeheader()
            for tag, name in mapping.items():
                writer.writerow({'anonymized_tag': tag, 'original_name': name})
        print(f"Mapping successfully saved to {filename}")
    except IOError as e:
        print(f"Error saving mapping to CSV: {e}")

def load_mapping_from_csv(filename: str) -> dict:
    """
    Loads an anonymization mapping from a CSV file.

    Args:
        filename (str): The path to the CSV file to load the mapping from.

    Returns:
        dict: The loaded dictionary mapping anonymized tags to original names.
    """
    mapping = {}
    try:
        if not os.path.exists(filename):
            print(f"CSV file not found: {filename}. Returning empty mapping.")
            return mapping

        with open(filename, 'r', newline='', encoding='utf-8') as csvfile:
            reader = csv.DictReader(csvfile)
            for row in reader:
                if 'anonymized_tag' in row and 'original_name' in row:
                    mapping[row['anonymized_tag']] = row['original_name']
                else:
                    print(f"Warning: Skipping row due to missing columns: {row}")
        print(f"Mapping successfully loaded from {filename}")
    except Exception as e:
        print(f"Error loading mapping from CSV: {e}")
        mapping = {} # Ensure mapping is empty if an error occurs
    return mapping
    

c:\Users\IbrahimaTHIOYE\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
# --- Example Usage ---
if __name__ == "__main__":
    sample_text_fr = "Emmanuel Macron est le président de la France. Il a rencontré Angela Merkel à Paris. Jean Dupont et Marie Curie étaient également présents. Madame Curie est une scientifique."
    mapping_filename = "anonymization_map.csv"

    print("Original Text:")
    print(sample_text_fr)
    print("-" * 30)

    # Anonymize the text
    anonymized_text, mapping = anonymize_text(sample_text_fr)

    print("\nAnonymized Text:")
    print(anonymized_text)
    print("\nAnonymization Mapping:")
    print(mapping)
    print("-" * 30)

    # Save the mapping to CSV
    save_mapping_to_csv(mapping, mapping_filename)

    # Load the mapping from CSV for demonstration
    loaded_mapping = load_mapping_from_csv(mapping_filename)
    print("\nLoaded Anonymization Mapping:")
    print(loaded_mapping)
    print("-" * 30)


    # Deanonymize the text using the loaded mapping
    deanonymized_text_reversed = deanonymize_text(anonymized_text, loaded_mapping)

    print("\nDeanonymized Text (Reversed using loaded mapping):")
    print(deanonymized_text_reversed)
    print("-" * 30)

    # Verify if the original text is recovered
    if sample_text_fr == deanonymized_text_reversed:
        print("\nVerification: Original text successfully recovered!")
    else:
        print("\nVerification: Mismatch between original and recovered text.")
        print("Original:", sample_text_fr)
        print("Recovered:", deanonymized_text_reversed)

    # Example with a name appearing multiple times
    text_with_repetition = "Dr. Dupont a parlé à Jean Dupont. Monsieur Dupont est son collègue."
    print("\n" + "="*30 + "\nExample with name repetition:")
    print("Original Text:")
    print(text_with_repetition)

    anon_rep_text, map_rep = anonymize_text(text_with_repetition)
    print("\nAnonymized Text:")
    print(anon_rep_text)
    print("\nAnonymization Mapping:")
    print(map_rep)

    # Save and load for this example too
    rep_mapping_filename = "anonymization_map_repetition.csv"
    save_mapping_to_csv(map_rep, rep_mapping_filename)
    loaded_map_rep = load_mapping_from_csv(rep_mapping_filename)

    dean_rep_text = deanonymize_text(anon_rep_text, loaded_map_rep)
    print("\nDeanonymized Text (Reversed):")
    print(dean_rep_text)

    if text_with_repetition == dean_rep_text:
        print("\nVerification (Repetition): Original text successfully recovered!")
    else:
        print("\nVerification (Repetition): Mismatch between original and recovered text.")

Original Text:
Emmanuel Macron est le président de la France. Il a rencontré Angela Merkel à Paris. Jean Dupont et Marie Curie étaient également présents. Madame Curie est une scientifique.
------------------------------


: 

In [1]:
from transformers import pipeline, AutoTokenizer
MODEL_NAME = "cmarkea/distilcamembert-base-ner"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
ner_pipeline = pipeline("ner", model=MODEL_NAME, tokenizer=tokenizer, aggregation_strategy="simple")

c:\Users\IbrahimaTHIOYE\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
c:\Users\IbrahimaTHIOYE\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\IbrahimaTHIOYE\.cache\huggingface\hub\models--cmarkea--distilcamembert-base-ner. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be di